<a href="https://colab.research.google.com/github/FadilSadBoy20/BigData26_A_2411532013_Fadil-Insanus-Siddik/blob/main/Praktikum02/BD_A_P02_2411532013_Fadil_Insanus_Siddik.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**LANGKAH PRAKTIKUM**

K-1. Import Library dan Inisialisasi

In [29]:
!pip install Faker

In [30]:
from google.colab import drive
drive.mount("/content/drive")

import os
DIR_KERJA   = "/content/data"
DIR_SIMPAN  = "/content/drive/MyDrive/BigData/Praktikum2"
os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print (os.listdir(DIR_SIMPAN))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['transaksi_bersih.csv', 'BD_A_P02_2411532013_Fadil_Insanus_Siddik.ipynb']


In [31]:
import numpy as np
import pandas as pd
from faker import Faker
import random

K-2. Membuat Dataset Sintetis (Simulasi Data Acquisition)

In [32]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
  trx_id = f"TR{i:05d}"
  nama_pelanggan = fake.name()
  produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
  kategori = random.choice(kategori_produk)
  harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
  qty = random.randint(1, 5)

  # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
  harga_variants = [
      str(harga_dasar),
      f"Rp{harga_dasar:,}".replace(",", "."),
      f"{harga_dasar}.0",
      f" {harga_dasar} ",
  ]
  harga = random.choice(harga_variants)

  # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYY
  tgl = fake.date_between(start_date="-90d", end_date="today")
  tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
  tanggal = random.choice(tgl_variants)

  metode = random.choice(metode_bayar)
  if random.random() < 0.3:
    metode = metode.lower()
  if random.random() < 0.2:
    kategori = kategori.upper() + " "

  kota = fake.city()
  rating = random.choice([1, 2, 3, 4, 5, None, None]) #rating opsional
  rows.append({
      "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
      "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
      "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
  })

  df = pd.DataFrame(rows)

  # Suntikkan missing value pada beberapa kolom
  for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index

    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang teratat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("jumlah baris:", len(df))

jumlah baris: 515


K-3. Deteksi dan Penanganan Missing Value

In [33]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


In [34]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

K-4. Deteksi dan Penanganan Duplicate

In [35]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


K-5. Koreksi Tipe Data dan Standarisasi Format

a. Standarisasi teks kategorikal

In [36]:
# Standardisasi teks kategorikal
for col in ["category", "payment_method", "shipping_city"]:
  df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan: kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

b. Koreksi tipe data pada kolom

In [37]:
# koreksi tipe data pada kolom price
def bersihkan_harga(x):
  if pd.isna(x):
    return np.nan
  x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
  try:
    return float(x)
  except ValueError:
    return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

c. Standarisasi format tanggal ke YYYY-MM-DD:

In [38]:
# standardisasi format tanggal ke yyyy-mm-dd
def parse_tanggal(x):
  for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
    try:
      return pd.to_datetime(x, format=fmt)
    except ValueError:
      continue
  return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

d. Finalisasi ke tipe data:

In [39]:
# finalisasi tipe data
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

Ekspor Dataset Bersih

In [40]:
df.to_csv(os.path.join(DIR_SIMPAN, "transaksi_bersih.csv"), index=False)
print("File berhasil disimpan ke Drive.")

File berhasil disimpan ke Drive.


**STUDI KASUS**

Platform marketplace kita mendeteksi kejanggalan pada laporan penjualan
bulanan: total transaksi yang dilaporkan tim IT (515) tidak sama dengan total yang dipakai
tim Finance (490). Sebagai calon data engineer, jelaskan kepada tim Finance:

1.

Mengapa kedua angka tersebut bisa berbeda, dikaitkan dengan proses yang baru saja
Anda lakukan.
Perbedaan jumlah transaksi terjadi karena adanya data duplikat pada dataset.
Berdasarkan hasil pengolahan, dataset mentah yang dibuat oleh Tim IT berjumlah 515
baris. Setelah dilakukan pengecekan menggunakan duplicated(), ditemukan 5 baris
yang memiliki data duplikat, termasuk transaction_id yang sama. Setelah data duplikat
dihapus menggunakan drop_duplicates(), jumlah transaksi menjadi 490 baris. Dengan
demikian, perbedaan antara 515 dan 490 berasal dari 25 baris yang tidak digunakan
dalam data akhir, yaitu data duplikat sebanyak 5 baris dan baris yang kemudian
tereliminasi
karena memiliki missing value pada customer_name atau
payment_method. Hasil isnull().sum() menunjukkan terdapat 20 missing value pada
customer_name dan 16 pada payment_method. Karena kode menggunakan
dropna(subset=["customer_name", "payment_method"]), baris yang memiliki nilai
kosong pada kedua kolom tersebut akan dihapus. Oleh karena itu, angka 515
menggambarkan jumlah data sebelum proses cleaning, sedangkan 490
menggambarkan jumlah data setelah sebagian proses pembersihan dilakukan.

2.

Apakah 490 baris “lebih benar” dibanding 515 baris? Jelaskan dengan mengaitkan ke
konsep Veracity.
Angka 490 tidak dapat langsung dianggap lebih benar hanya karena jumlah datanya
lebih sedikit. Dalam konsep Veracity, yang diperhatikan adalah tingkat kebenaran,
keandalan, dan kualitas data yang digunakan. Lima baris yang teridentifikasi sebagai
duplikat memang dapat dianggap sebagai data yang perlu dibersihkan apabila terbukti
merupakan transaksi yang tercatat dua kali. Namun, penghapusan data yang memiliki
missing value juga merupakan keputusan pengolahan data dan tidak selalu berarti
bahwa transaksi tersebut tidak valid. Misalnya, transaksi yang tidak memiliki
customer_name atau payment_method masih mungkin merupakan transaksi yang
benar-benar terjadi. Oleh karena itu, 490 merupakan jumlah data setelah aturan
cleaning diterapkan, bukan secara otomatis jumlah transaksi yang paling benar. Untuk
memastikan veracity, diperlukan pemeriksaan lebih lanjut terhadap sumber data
transaksi dan alasan mengapa data tersebut memiliki nilai kosong atau tercatat
duplikat.

3.

Bagaimana Anda akan menjelaskan keputusan membiarkan kolom rating tetap
memiliki missing value kepada tim Finance yang ingin tahu “rating rata-rata semua
transaksi”?
Missing value pada kolom rating sengaja tidak dihapus karena rating merupakan data
yang bersifat opsional. Berdasarkan hasil isnull().sum(), terdapat 166 transaksi yang
tidak memiliki rating, sehingga hanya sebagian transaksi yang memberikan penilaian.
Jika baris tersebut dihapus hanya karena tidak memiliki rating, maka jumlah transaksi
untuk analisis lain juga akan berkurang dan dapat menyebabkan kehilangan informasi.
Untuk menghitung rating rata-rata, nilai yang kosong tidak seharusnya dianggap
sebagai nilai 0, karena tidak memberikan rating bukan berarti pelanggan memberikan
penilaian nol. Oleh karena itu, rata-rata rating sebaiknya dihitung hanya dari transaksi
yang memiliki nilai rating, kemudian dilaporkan bersama jumlah transaksi yang
memiliki rating. Dengan cara tersebut, Finance dapat memahami bahwa rata-rata
tersebut merupakan rata-rata dari transaksi yang memberikan rating, bukan rata-rata
dari seluruh 490 transaksi.

**LATIHAN**

In [41]:
SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
  trx_id = f"TR{i:05d}"
  nama_pelanggan = fake.name()
  produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
  kategori = random.choice(kategori_produk)
  harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
  qty = random.randint(1, 5)

  # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
  harga_variants = [
      str(harga_dasar),
      f"Rp{harga_dasar:,}".replace(",", "."),
      f"{harga_dasar}.0",
      f" {harga_dasar} ",
  ]
  harga = random.choice(harga_variants)

  # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYY
  tgl = fake.date_between(start_date="-90d", end_date="today")
  tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
  tanggal = random.choice(tgl_variants)

  metode = random.choice(metode_bayar)
  if random.random() < 0.3:
    metode = metode.lower()
  if random.random() < 0.2:
    kategori = kategori.upper() + " "

  kota = fake.city()
  rating = random.choice([1, 2, 3, 4, 5, None, None]) #rating opsional
  rows.append({
      "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
      "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
      "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
  })

  df = pd.DataFrame(rows)

  # Suntikkan missing value pada beberapa kolom
  for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index

    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang teratat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("jumlah baris:", len(df))

jumlah baris: 515


In [42]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              120
dtype: int64


In [43]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

In [44]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


In [45]:
# Standardisasi teks kategorikal
for col in ["category", "payment_method", "shipping_city"]:
  df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan: kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

In [46]:
# koreksi tipe data pada kolom price
def bersihkan_harga(x):
  if pd.isna(x):
    return np.nan
  x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
  try:
    return float(x)
  except ValueError:
    return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

In [47]:
# standardisasi format tanggal ke yyyy-mm-dd
def parse_tanggal(x):
  for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
    try:
      return pd.to_datetime(x, format=fmt)
    except ValueError:
      continue
  return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

In [48]:
# finalisasi tipe data
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

# Menambahkan kolom validasi harga
df["is_valid_price"] = df["price"] > 0

# Menghitung jumlah transaksi per kategori
jumlah_transaksi_kategori = df["category"].value_counts()

print("Jumlah transaksi per kategori:")
print(jumlah_transaksi_kategori)

Jumlah transaksi per kategori:
category
Rumah Tangga    91
Kesehatan       86
Buku            81
Fashion         80
Elektronik      78
Olahraga        74
Name: count, dtype: Int64
